# E-Commerce Logistics & Sales Analyzer
* The Scenario: You are given three separate datasets: "Customer Profiles" (age, region, subscription tier), "Product Catalog" (category, weight, unit price), and "Order Transactions" (who bought what, when, and delivery times).
* Combining Datasets: You must use pd.concat() to stitch together the order transactions from "January" and "February". Then, use pd.merge() to join the combined 'Order Transactions' table with the 'Product Catalog' table so you know the category and price of the item purchased, acting like a SQL left join.
* Modifying DataFrames: You create a new calculated column called Total_Revenue by multiplying the Quantity column by the Unit_Price column. Use .drop() to remove redundant warehouse routing IDs, and use .rename() to fix inconsistently capitalized columns (e.g., changing cust_ID to Customer_ID).
* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the Delivery_Days column into delivery performance buckets: "Early" (1-2 days), "On-Time" (3-4 days), or "Delayed" (5+ days).
* Grouping and Aggregation: Using .groupby(), you group the data by Product_Category and use .agg() to find the total revenue generated, the average delivery days, and the unique count of Customer_IDs who purchased from that category.
* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a business-reporting matrix showing "Customer Region" as rows, "Delivery Performance" (Early/On-Time/Delayed) as columns, and the "Total Revenue" as the values.

* Combining Datasets: You must use pd.concat() to stitch together the order transactions from "January" and "February". Then, use pd.merge() to join the combined 'Order Transactions' table with the 'Product Catalog' table so you know the category and price of the item purchased, acting like a SQL left join.


In [101]:
import numpy as np 
import pandas as pd
df_customer = pd.read_csv('customers - customers.csv')
df_order_feb = pd.read_csv('orders_feb - orders_feb.csv')
df_order_jan = pd.read_csv('orders_jan - orders_jan.csv')
df_products = pd.read_csv('products - products.csv')
order_transations = pd.concat([df_order_feb, df_order_jan] , ignore_index = True)
# order_transations
df = pd.merge(order_transations , df_products, on= 'Product_ID' , how= 'left')
df

,Transaction_ID,cust_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Warehouse_Route_ID,Product_Category,Weight_kg,Unit_Price
0,T01001,C0942,P0525,5,2024-02-13,6,WH-D9,Home & Garden,5.59,35.71
1,T01002,C0675,P0908,3,2024-02-08,7,WH-A1,Sports,4.39,128.06
2,T01003,C0685,P0255,4,2024-02-19,3,WH-A3,Sports,9.60,284.14
3,T01004,C0141,P0085,4,2024-02-05,5,WH-C8,Electronics,4.05,379.86
4,T01005,C0138,P0458,2,2024-02-02,7,WH-E8,Toys,0.22,438.19
...,...,...,...,...,...,...,...,...,...,...
1995,T00996,C0196,P0282,3,2024-01-19,7,WH-B2,Electronics,11.09,165.15
1996,T00997,C0307,P0133,3,2024-01-24,2,WH-D6,Sports,10.80,172.04
1997,T00998,C0664,P0534,5,2024-01-15,3,WH-C9,Electronics,10.04,26.81
1998,T00999,C0014,P0975,5,2024-01-26,5,WH-C3,Toys,5.78,399.24


* Modifying DataFrames: You create a new calculated column called Total_Revenue by multiplying the Quantity column by the Unit_Price column. Use .drop() to remove redundant warehouse routing IDs, and use .rename() to fix inconsistently capitalized columns (e.g., changing cust_ID to Customer_ID).


In [102]:
df['Total_Revenue'] = df['Quantity'] * df['Unit_Price']
df.drop('Warehouse_Route_ID',axis = 1 , inplace= True, errors= 'ignore')
df.rename(columns={'cust_ID': 'Customer_ID'}, inplace= True)
df = pd.merge(df_customer, df, on= 'Customer_ID' ,how= 'left')
df

,Customer_ID,Age,Region,Subscription_Tier,Transaction_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Product_Category,Weight_kg,Unit_Price,Total_Revenue
0,C0001,56,North,Basic,T01112,P0417,4.0,2024-02-18,1.0,Sports,10.88,79.96,319.84
1,C0001,56,North,Basic,T01267,P0623,5.0,2024-02-11,6.0,Toys,7.38,298.92,1494.60
2,C0001,56,North,Basic,T01543,P0938,4.0,2024-02-29,3.0,Toys,11.11,152.34,609.36
3,C0001,56,North,Basic,T00015,P0335,2.0,2024-01-15,2.0,Toys,4.69,377.97,755.94
4,C0002,69,South,Plus,T00378,P0349,1.0,2024-01-05,4.0,Sports,10.19,58.48,58.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2118,C0998,22,North,Premium,T00932,P0616,3.0,2024-01-14,3.0,Clothing,11.71,495.81,1487.43
2119,C0998,22,North,Premium,T00987,P0864,4.0,2024-01-23,6.0,Sports,3.99,460.74,1842.96
2120,C0999,51,East,Basic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2121,C1000,23,West,Basic,T00286,P0557,2.0,2024-01-21,1.0,Home & Garden,0.43,336.86,673.72


* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the Delivery_Days column into delivery performance buckets: "Early" (1-2 days), "On-Time" (3-4 days), or "Delayed" (5+ days).


In [107]:
def time_delivery(time) :
    if time <=2 :
        return 'Early'
    elif time <=4 :
        return 'On_Time'
    else :
        return 'Delayed'
df['Time_arrival'] = df['Delivery_Days'].apply(time_delivery)
df

,Customer_ID,Age,Region,Subscription_Tier,Transaction_ID,Product_ID,Quantity,Order_Date,Delivery_Days,Product_Category,Weight_kg,Unit_Price,Total_Revenue,Time_arrival
0,C0001,56,North,Basic,T01112,P0417,4.0,2024-02-18,1.0,Sports,10.88,79.96,319.84,Early
1,C0001,56,North,Basic,T01267,P0623,5.0,2024-02-11,6.0,Toys,7.38,298.92,1494.60,Delayed
2,C0001,56,North,Basic,T01543,P0938,4.0,2024-02-29,3.0,Toys,11.11,152.34,609.36,On_Time
3,C0001,56,North,Basic,T00015,P0335,2.0,2024-01-15,2.0,Toys,4.69,377.97,755.94,Early
4,C0002,69,South,Plus,T00378,P0349,1.0,2024-01-05,4.0,Sports,10.19,58.48,58.48,On_Time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2118,C0998,22,North,Premium,T00932,P0616,3.0,2024-01-14,3.0,Clothing,11.71,495.81,1487.43,On_Time
2119,C0998,22,North,Premium,T00987,P0864,4.0,2024-01-23,6.0,Sports,3.99,460.74,1842.96,Delayed
2120,C0999,51,East,Basic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Delayed
2121,C1000,23,West,Basic,T00286,P0557,2.0,2024-01-21,1.0,Home & Garden,0.43,336.86,673.72,Early


* Grouping and Aggregation: Using .groupby(), you group the data by Product_Category and use .agg() to find the total revenue generated, the average delivery days, and the unique count of Customer_IDs who purchased from that category.


In [66]:
analy_df_agg = df.groupby('Product_Category').agg({
    'Delivery_Days': ['sum', 'mean'],
    'Customer_ID' : 'nunique'
})
analy_df_agg

Delivery_Days           Customer_ID
                           sum      mean     nunique
Product_Category                                    
Clothing                  1390  4.005764         289
Electronics               1612  3.960688         333
Home & Garden             1720  4.009324         342
Sports                    1612  3.847255         346
Toys                      1541  3.871859         316

* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a business-reporting matrix showing "Customer Region" as rows, "Delivery Performance" (Early/On-Time/Delayed) as columns, and the "Total Revenue" as the values.

In [104]:
business_reporting = pd.pivot_table(
    df,
    values = 'Total_Revenue',
    index = 'Region',
    columns = 'Delivery_Days',
    aggfunc = 'sum'
)
business_reporting

Delivery_Days,1.0,2.0,3.0,4.0,5.0,6.0,7.0
Region,,,,,,,
East,51222.77,40273.06,62933.93,39895.37,61624.16,61279.01,42629.65
North,62955.73,52888.07,65787.32,59591.54,54701.29,50760.99,63450.89
South,51552.61,60586.97,54533.07,48107.19,53380.80,51278.88,52872.26
West,68166.47,43336.88,35417.86,40160.92,45251.39,40491.02,43319.01
